# Задание

Ваша задача - реализовать класс Account, который моделирует поведение банковского счёта. Этот класс должен не только выполнять базовые операции, но и вести детальный учёт всех действий, а также предоставлять аналитику по истории операций.

## Этап 1. Реализация базового класса Account

Класс должен быть инициализирован с параметрами:

- account_holder (str): имя владельца счёта;
- balance (float, по умолчанию 0): начальный баланс счёта, не может быть отрицательным.

Атрибуты:

- holder: хранит имя владельца;
- _balance: приватный атрибут для хранения текущего баланса;
- operations_history: список или другая структура для хранения истории операций.

Важно: каждая операция должна храниться не просто как число, а как структурированная информация, например, словарь или кортеж. Минимальный набор данных для операции: тип операции ('deposit' или 'withdraw'), сумма, дата и время операции, текущий баланс после операции, статус ('success' или 'fail').

## Этап 2. Реализация методов

- __init__(self, account_holder, balance=0): конструктор;
- deposit(self, amount): метод для пополнения счёта:
  - принимает сумму (должна быть положительной);
  - в случае успеха обновляет баланс и добавляет запись в историю операций.
- withdraw(self, amount): метод для снятия средств:
  - принимает сумму (должна быть положительной);
  - проверяет, достаточно ли средств на счёте, если нет — операция не проходит, но ее попытка с статусом 'fail' все равно фиксируется в истории;
  - в случае успеха обновляет баланс и добавляет запись.
- get_balance(self): метод, который возвращает текущий баланс.
- get_history(self): метод, который возвращает историю операций.

Важно: продумайте, в каком формате его вернуть. Для работы с датой и временем используйте модуль datetime. Получить текущее время можно с помощью datetime.now().

## Этап 3. Реализация наследования

Создайте класс CreditAccount(Account), который наследует всю функциональность класса Account и добавляет новую.

Особенности кредитного счёта:

- При инициализации принимает дополнительный параметр credit_limit (кредитный лимит). Баланс такого счета может быть отрицательным, но не ниже значения -credit_limit.
- По запросу показывает, сколько кредитных средств еще доступно (текущий баланс + кредитный лимит).
- В историю операций добавляется информация о том, были ли использованы кредитные средства в данной операции.

### Решение

In [1]:
from copy import deepcopy
from datetime import datetime


class Account:
    def __init__(self, account_holder, balance=0):
        if not isinstance(account_holder, str) or not account_holder.strip():
            raise ValueError("account_holder must be a non-empty string")
        if not self._is_number(balance):
            raise TypeError("balance must be a number")
        if balance < 0:
            raise ValueError("initial balance cannot be negative")

        self.holder = account_holder.strip()
        self._balance = float(balance)
        self.operations_history = []

    @staticmethod
    def _is_number(value):
        # bool is a subclass of int, reject it explicitly
        return isinstance(value, (int, float)) and not isinstance(value, bool)

    def _is_valid_amount(self, amount):
        return self._is_number(amount) and amount > 0

    def get_balance(self):
        return self._balance

    def get_history(self):
        # return a copy so callers cannot mutate internal state
        return deepcopy(self.operations_history)

    def _add_operation(self, op_type, amount, status, used_credit=None, note=""):
        record = {
            "type": op_type,
            "amount": float(amount),
            "timestamp": datetime.now(),
            "balance_after": self._balance,
            "status": status,
        }
        if used_credit is not None:
            record["used_credit"] = used_credit
        if note:
            record["note"] = note
        self.operations_history.append(record)
        return record

    def deposit(self, amount):
        if not self._is_valid_amount(amount):
            self._add_operation(
                "deposit",
                amount if self._is_number(amount) else 0,
                "fail",
                note="invalid amount",
            )
            return False

        self._balance += float(amount)
        self._add_operation("deposit", amount, "success")
        return True

    def withdraw(self, amount):
        if not self._is_valid_amount(amount):
            self._add_operation(
                "withdraw",
                amount if self._is_number(amount) else 0,
                "fail",
                note="invalid amount",
            )
            return False

        amount = float(amount)
        if amount > self._balance:
            # failed attempt is still recorded
            self._add_operation("withdraw", amount, "fail", note="insufficient funds")
            return False

        self._balance -= amount
        self._add_operation("withdraw", amount, "success")
        return True

    def __str__(self):
        return f"{self.holder}: {self._balance:.2f}"

In [2]:
class CreditAccount(Account):
    def __init__(self, account_holder, balance=0, credit_limit=0):
        if not self._is_number(credit_limit) or credit_limit < 0:
            raise ValueError("credit_limit must be a number >= 0")
        super().__init__(account_holder, balance)
        self.credit_limit = float(credit_limit)

    def get_available_credit(self):
        return self._balance + self.credit_limit

    def withdraw(self, amount):
        if not self._is_valid_amount(amount):
            self._add_operation(
                "withdraw",
                amount if self._is_number(amount) else 0,
                "fail",
                used_credit=False,
                note="invalid amount",
            )
            return False

        amount = float(amount)
        new_balance = self._balance - amount

        if new_balance < -self.credit_limit:
            self._add_operation(
                "withdraw",
                amount,
                "fail",
                used_credit=False,
                note="credit limit exceeded",
            )
            return False

        used_credit = new_balance < 0
        self._balance = new_balance
        self._add_operation("withdraw", amount, "success", used_credit=used_credit)
        return True

    def deposit(self, amount):
        if not self._is_valid_amount(amount):
            self._add_operation(
                "deposit",
                amount if self._is_number(amount) else 0,
                "fail",
                used_credit=self._balance < 0,
                note="invalid amount",
            )
            return False

        was_in_credit = self._balance < 0
        self._balance += float(amount)
        self._add_operation("deposit", amount, "success", used_credit=was_in_credit)
        return True

    def __str__(self):
        return (
            f"{self.holder}: {self._balance:.2f} "
            f"(limit {self.credit_limit:.2f}, available {self.get_available_credit():.2f})"
        )

### Helpers

In [3]:
def print_history(account):
    history = account.get_history()
    if not history:
        print("history is empty")
        return

    for i, op in enumerate(history, start=1):
        when = op["timestamp"].strftime("%Y-%m-%d %H:%M:%S")
        line = (
            f"{i}. {when} | {op['type']} {op['amount']:.2f} | "
            f"balance={op['balance_after']:.2f} | {op['status']}"
        )
        if "used_credit" in op:
            line += f" | used_credit={op['used_credit']}"
        if op.get("note"):
            line += f" | {op['note']}"
        print(line)


def summary(account):
    history = account.get_history()
    ok_in = [o for o in history if o["type"] == "deposit" and o["status"] == "success"]
    ok_out = [o for o in history if o["type"] == "withdraw" and o["status"] == "success"]
    fails = [o for o in history if o["status"] == "fail"]
    return {
        "ops": len(history),
        "deposits": len(ok_in),
        "withdrawals": len(ok_out),
        "fails": len(fails),
        "sum_in": sum(o["amount"] for o in ok_in),
        "sum_out": sum(o["amount"] for o in ok_out),
        "balance": account.get_balance(),
    }


def filter_history(account, op_type=None, status=None):
    result = []
    for op in account.get_history():
        if op_type is not None and op["type"] != op_type:
            continue
        if status is not None and op["status"] != status:
            continue
        result.append(op)
    return result


def transfer(from_acc, to_acc, amount):
    if from_acc is to_acc:
        return False
    if not from_acc.withdraw(amount):
        return False
    if not to_acc.deposit(amount):
        # rare path: roll back sender if receiver deposit failed
        from_acc._balance += float(amount)
        from_acc._add_operation("deposit", amount, "success", note="transfer rollback")
        return False
    return True

### Для ручной проверки

In [4]:
# @title Ordinary account
holder = "Anna"  # @param {type:"string"}
start_balance = 1000  # @param {type:"number"}

acc = Account(holder, start_balance)
print(acc)
print("balance =", acc.get_balance())

Anna: 1000.00
balance = 1000.0


In [5]:
# @title Ordinary account operations
deposit_sum = 200  # @param {type:"number"}
withdraw_sum = 50  # @param {type:"number"}
too_much = 99999  # @param {type:"number"}

print("deposit:", acc.deposit(deposit_sum))
print("withdraw:", acc.withdraw(withdraw_sum))
print("withdraw too much:", acc.withdraw(too_much))
print()
print_history(acc)
print()
print(summary(acc))

deposit: True
withdraw: True
withdraw too much: False

1. 2026-10-24 15:38:16 | deposit 200.00 | balance=1200.00 | success
2. 2026-10-24 15:38:16 | withdraw 50.00 | balance=1150.00 | success
3. 2026-10-24 15:38:16 | withdraw 99999.00 | balance=1150.00 | fail | insufficient funds

{'ops': 3, 'deposits': 1, 'withdrawals': 1, 'fails': 1, 'sum_in': 200.0, 'sum_out': 50.0, 'balance': 1150.0}


In [6]:
# @title Credit account
credit_holder = "Boris"  # @param {type:"string"}
credit_balance = 100  # @param {type:"number"}
credit_limit = 300  # @param {type:"number"}

cred = CreditAccount(credit_holder, credit_balance, credit_limit)
print(cred)
print("available =", cred.get_available_credit())

Boris: 100.00 (limit 300.00, available 400.00)
available = 400.0


In [7]:
# @title Credit account operations
credit_withdraw = 250  # @param {type:"number"}
credit_deposit = 40  # @param {type:"number"}
over_limit = 1000  # @param {type:"number"}

print("withdraw:", cred.withdraw(credit_withdraw))
print("balance =", cred.get_balance(), "| available =", cred.get_available_credit())
print("deposit:", cred.deposit(credit_deposit))
print("over limit:", cred.withdraw(over_limit))
print()
print_history(cred)

withdraw: True
balance = -150.0 | available = 150.0
deposit: True
over limit: False

1. 2026-10-24 15:38:17 | withdraw 250.00 | balance=-150.00 | success | used_credit=True
2. 2026-10-24 15:38:17 | deposit 40.00 | balance=-110.00 | success | used_credit=True
3. 2026-10-24 15:38:17 | withdraw 1000.00 | balance=-110.00 | fail | used_credit=False | credit limit exceeded


In [8]:
# @title Transfer
from_name = "Anna"  # @param {type:"string"}
to_name = "Boris"  # @param {type:"string"}
transfer_amount = 30  # @param {type:"number"}

if acc.holder != from_name:
    print(f"unknown sender: {from_name!r} (expected {acc.holder!r})")
elif cred.holder != to_name:
    print(f"unknown receiver: {to_name!r} (expected {cred.holder!r})")
else:
    print("before:", acc.get_balance(), "->", cred.get_balance())
    print("transfer:", transfer(acc, cred, transfer_amount))
    print("after:", acc.get_balance(), "->", cred.get_balance())

before: 1150.0 -> -110.0
transfer: True
after: 1120.0 -> -80.0


### Тесты

In [9]:
# basic flow
a = Account("Katya", 100)
assert a.deposit(50) is True
assert a.get_balance() == 150
assert a.withdraw(20) is True
assert a.get_balance() == 130

# failed withdraw must still appear in history
assert a.withdraw(10_000) is False
assert a.get_balance() == 130
assert a.get_history()[-1]["status"] == "fail"

# edge cases (part of)
assert a.deposit(0) is False
assert a.deposit(-1) is False
assert a.withdraw(0) is False
assert a.withdraw(-5) is False
assert a.deposit(True) is False   # bool subclasses int
assert a.withdraw(False) is False

# withdraw exact balance
assert a.withdraw(a.get_balance()) is True
assert a.get_balance() == 0

# history shape + timestamps
for op in a.get_history():
    assert {"type", "amount", "timestamp", "balance_after", "status"} <= set(op)
    assert isinstance(op["timestamp"], datetime)

# get_history returns a copy, not the internal list
h = a.get_history()
h.clear()
assert len(a.get_history()) > 0

# invalid construction
try:
    Account("X", -10)
    assert False
except ValueError:
    pass

try:
    Account("   ")
    assert False
except ValueError:
    pass

# credit account
c = CreditAccount("Max", 100, credit_limit=200)
assert isinstance(c, Account)
assert c.withdraw(250) is True
assert c.get_balance() == -150
assert c.get_available_credit() == 50
assert c.get_history()[-1]["used_credit"] is True

# exactly at credit limit
c2 = CreditAccount("Limit", 0, credit_limit=100)
assert c2.withdraw(100) is True
assert c2.get_balance() == -100
assert c2.get_available_credit() == 0

# one cent over the limit
assert c2.withdraw(0.01) is False
assert c2.get_history()[-1]["status"] == "fail"
assert c2.get_balance() == -100

# deposit while balance is negative
assert c.deposit(50) is True
assert c.get_balance() == -100
assert c.get_history()[-1]["used_credit"] is True

# transfer
x = Account("A", 80)
y = Account("B", 10)
assert transfer(x, y, 30) is True
assert x.get_balance() == 50 and y.get_balance() == 40
assert transfer(x, x, 1) is False

print("all tests passed")

all tests passed


### Небольшой бонус: гонка при одновременном снятии

`withdraw` сначала проверяет баланс, потом списывает. При одновременном снятии баланс может уйти в минус

Тут добавлена пауза между проверкой и списанием, чтобы гонку было проще увидеть. Потом то же самое, но под `Lock`. Без паузы GIL ловит.

In [10]:
import threading
import time


def unsafe_withdraw(account, amount, pause=0.05):
    if not account._is_valid_amount(amount):
        account._add_operation(
            "withdraw",
            amount if account._is_number(amount) else 0,
            "fail",
            note="invalid amount",
        )
        return False

    amount = float(amount)
    if amount > account._balance:
        account._add_operation("withdraw", amount, "fail", note="insufficient funds")
        return False

    time.sleep(pause)
    account._balance -= amount
    account._add_operation("withdraw", amount, "success")
    return True


def run_parallel_withdraws(make_withdraw, balance=100, amount=80):
    account = Account("RaceDemo", balance)
    results = []

    def worker():
        results.append(make_withdraw(account, amount))

    threads = [threading.Thread(target=worker) for _ in range(2)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    return account, results


broken_acc, broken_results = run_parallel_withdraws(unsafe_withdraw)
print("без лока")
print("  результаты:", broken_results)
print("  баланс:", broken_acc.get_balance())

lock = threading.Lock()


def safe_withdraw(account, amount, pause=0.05):
    with lock:
        return unsafe_withdraw(account, amount, pause=pause)


safe_acc, safe_results = run_parallel_withdraws(safe_withdraw)
print()
print("с локом")
print("  результаты:", safe_results)
print("  баланс:", safe_acc.get_balance())

без лока
  результаты: [True, True]
  баланс: -60.0

с локом
  результаты: [True, False]
  баланс: 20.0
